# Embarrassingly parallel R on Snowflake (SPCS + doSnowflake)

Forecasting here is only an **illustration**; the same patterns apply to HPO grids, back-tests, Monte Carlo, etc.

**Prerequisites:** `workspace_parallel_spcs_setup.ipynb` once.

**Config:** `parallel_lab` + optional `context` in `snowflaker_parallel_spcs_config.yaml` (via `parallel_lab_config.py`).
**Dev default:** YAML points at **MR Price** warehouse / pool / DB already used by internal doSnowflake benchmarks; swap to generic names before publishing.
**Run gate:** after §2, use a **scratch R cell** `RUN_PARALLEL_DEMO <<- TRUE` when you want §3–§5 to run `foreach` / registry work (keeps git clean).
**Scale:** the §1 Python setup cell prints `demo_*` from `parallel_lab` (defaults match `test_dosnowflake_tasks_benchmark.py`: 2000 SKUs, 10 chunks).
**Monitoring:** driver cells block — use `workspace_parallel_spcs_monitor.ipynb` or Streamlit in another tab.


## 1. Environment


In [ ]:
from sfnb_setup import setup_notebook
setup_notebook(
    config="snowflaker_parallel_spcs_config.yaml",
    packages=["snowflakeR", "RSnowflake"]
)
import parallel_lab_config as plc
plc.ensure_notebook_path_on_sys_path()
LAB = plc.apply_parallel_lab_environment(plc.load_parallel_lab_dict())
print("parallel_lab:", LAB["database"], "| pool:", LAB.get("compute_pool") or "(unset)")
print(
    "demo:", LAB.get("demo_forecast_n_skus"), "SKUs | tasks chunks:",
    LAB.get("demo_tasks_chunks_per_job"), "| queue workers:",
    LAB.get("demo_queue_n_workers"),
)


## 2. Connect (R)


In [ ]:
%%R
library(snowflakeR)
library(dplyr)
library(dbplyr)
library(RSnowflake)
ep_db <- Sys.getenv("PARALLEL_LAB_DATABASE")
sch_src <- Sys.getenv("PARALLEL_LAB_SCHEMA_SOURCE_DATA")
wh <- Sys.getenv("PARALLEL_LAB_WAREHOUSE")
conn <- sfr_connect()
if (nzchar(wh)) {
  sfr_execute(conn, paste("USE WAREHOUSE", wh))
}
conn <- sfr_use(conn, database = ep_db, schema = sch_src)
cat("Using", ep_db, ".", sch_src, "\n", sep = "")
# Default FALSE: scratch R cell `RUN_PARALLEL_DEMO <<- TRUE` before §3–§5.
RUN_PARALLEL_DEMO <- FALSE


## 2a. Pre-warm SPCS compute pool

Run this right after connect so **`parallel_lab.compute_pool`** starts **RESUME** while you read ahead or set the run gate (same idea as `workspace_mr_price_r_demo.ipynb`).


In [ ]:
%%R
pool <- Sys.getenv("PARALLEL_LAB_COMPUTE_POOL")
if (!nzchar(pool)) {
  stop("PARALLEL_LAB_COMPUTE_POOL is unset; check parallel_lab in YAML")
}
sfr_execute(conn, sprintf("ALTER COMPUTE POOL %s RESUME IF SUSPENDED", pool))
pst <- sfr_query(conn, sprintf("SHOW COMPUTE POOLS LIKE '%s'", pool), .keep_case = FALSE)
cat("Pool", pool, "state:", pst$state[[1]], "\n", sep = " ")


## 3. Static work allocation (`doSnowflake` tasks mode)

Registers **tasks** mode then runs a **`foreach` × `forecast`** loop over **`demo_forecast_n_skus`** synthetic SKUs (workload body matches `test_dosnowflake_tasks_benchmark.py`). Parallelism = **`demo_tasks_chunks_per_job`** SPCS jobs.

Gated by **`RUN_PARALLEL_DEMO`** (§2). Tune **`demo_*`** keys under `parallel_lab` in YAML.


In [ ]:
%%R
pool   <- Sys.getenv("PARALLEL_LAB_COMPUTE_POOL")
img    <- Sys.getenv("PARALLEL_LAB_IMAGE_URI")
stage  <- Sys.getenv("PARALLEL_LAB_STAGE_AT")
n_skus <- as.integer(Sys.getenv("PARALLEL_LAB_DEMO_FORECAST_N_SKUS"))
n_ch   <- as.integer(Sys.getenv("PARALLEL_LAB_DEMO_TASKS_CHUNKS_PER_JOB"))
run    <- isTRUE(RUN_PARALLEL_DEMO)
chunk_sz <- max(1L, as.integer(ceiling(n_skus / n_ch)))
cat(
  "doSnowflake tasks:\n",
  "  compute_pool =", pool,
  "\n  image_uri =", img,
  "\n  stage =", stage,
  "\n  n_skus =", n_skus, "  chunks_per_job =", n_ch,
  "  (~", chunk_sz, " SKUs/chunk)\n",
  "  RUN_PARALLEL_DEMO =", run, "\n",
  sep = ""
)

if (!run) {
  cat("Skipping registerDoSnowflake + foreach (tasks). Scratch: RUN_PARALLEL_DEMO <<- TRUE\n")
} else {
  library(foreach)
  registerDoSnowflake(
    conn,
    mode           = "tasks",
    compute_pool   = pool,
    image_uri      = img,
    chunks_per_job = n_ch
  )
  skus <- sprintf("SKU_DEMO_%05d", seq_len(n_skus))
  t0 <- Sys.time()
  out <- foreach(
    sku = skus,
    .combine = list,
    .multicombine = TRUE,
    .packages = "forecast"
  ) %dopar% {
    suppressPackageStartupMessages(library(forecast))
    seed_str <- gsub("[^0-9]", "", sku)
    if (nchar(seed_str) == 0) seed_str <- "42"
    set.seed(as.integer(substr(seed_str, 1, 8)) %% 10000)
    n <- 36L
    trend <- seq(100, 200, length.out = n)
    seasonal <- 20 * sin(2 * pi * (1:n) / 12)
    noise <- rnorm(n, 0, 10)
    quantities <- pmax(round(trend + seasonal + noise), 1)
    ts_data <- ts(quantities, frequency = 12)
    model <- auto.arima(ts_data)
    fc <- forecast(model, h = 6)
    list(
      sku      = sku,
      model    = as.character(fc$method),
      forecast = as.numeric(fc$mean)
    )
  }
  dt <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
  cat("Finished", length(out), "forecasts in", round(dt, 1), "s\n")
  cat("Example:", out[[1]]$sku, out[[1]]$model, "fc head:", head(out[[1]]$forecast), "\n")
}


## 4. Elastic queue (`doSnowflake` queue mode)

Same **`RUN_PARALLEL_DEMO`** gate and **SKU / forecast** body as §3. **`demo_queue_n_workers`** ephemeral workers, **`demo_queue_chunks_per_job`** chunks. Re-registering switches the foreach backend from tasks to queue.


In [ ]:
%%R
n_skus <- as.integer(Sys.getenv("PARALLEL_LAB_DEMO_FORECAST_N_SKUS"))
n_wrk  <- as.integer(Sys.getenv("PARALLEL_LAB_DEMO_QUEUE_N_WORKERS"))
n_qch  <- as.integer(Sys.getenv("PARALLEL_LAB_DEMO_QUEUE_CHUNKS_PER_JOB"))
run    <- isTRUE(RUN_PARALLEL_DEMO)
cat(
  "doSnowflake queue:\n",
  "  queue FQN =", Sys.getenv("PARALLEL_LAB_QUEUE_FQN"),
  "\n  n_skus =", n_skus, "  n_workers =", n_wrk, "  chunks_per_job =", n_qch,
  "\n  RUN_PARALLEL_DEMO =", run, "\n",
  sep = ""
)

if (!run) {
  cat("Skipping registerDoSnowflake + foreach (queue). Scratch: RUN_PARALLEL_DEMO <<- TRUE\n")
} else {
  library(foreach)
  registerDoSnowflake(
    conn,
    mode            = "queue",
    worker_type     = "ephemeral",
    compute_pool    = Sys.getenv("PARALLEL_LAB_COMPUTE_POOL"),
    image_uri       = Sys.getenv("PARALLEL_LAB_IMAGE_URI"),
    n_workers       = n_wrk,
    chunks_per_job  = n_qch
  )
  skus <- sprintf("SKU_DEMO_%05d", seq_len(n_skus))
  t0 <- Sys.time()
  outq <- foreach(
    sku = skus,
    .combine = list,
    .multicombine = TRUE,
    .packages = "forecast"
  ) %dopar% {
    suppressPackageStartupMessages(library(forecast))
    seed_str <- gsub("[^0-9]", "", sku)
    if (nchar(seed_str) == 0) seed_str <- "42"
    set.seed(as.integer(substr(seed_str, 1, 8)) %% 10000)
    n <- 36L
    trend <- seq(100, 200, length.out = n)
    seasonal <- 20 * sin(2 * pi * (1:n) / 12)
    noise <- rnorm(n, 0, 10)
    quantities <- pmax(round(trend + seasonal + noise), 1)
    ts_data <- ts(quantities, frequency = 12)
    model <- auto.arima(ts_data)
    fc <- forecast(model, h = 6)
    list(
      sku      = sku,
      model    = as.character(fc$method),
      forecast = as.numeric(fc$mean)
    )
  }
  dt <- as.numeric(difftime(Sys.time(), t0, units = "secs"))
  cat("Queue mode: finished", length(outq), "forecasts in", round(dt, 1), "s\n")
}


## 5. Model Registry & batch inference

Registry schema: **`Sys.getenv("PARALLEL_LAB_SCHEMA_MODELS")`** under **`Sys.getenv("PARALLEL_LAB_DATABASE")`**.

Bundled partitions for batch inference — see `PARALLEL_SPCS_DEMO.md`.


In [ ]:
%%R
run <- isTRUE(RUN_PARALLEL_DEMO)
ep_models <- Sys.getenv("PARALLEL_LAB_SCHEMA_MODELS")
cat("Model registry target:", ep_db, ".", ep_models,
    " | RUN_PARALLEL_DEMO =", run, "\n", sep = "")

if (!run) {
  cat("Skipping sfr_model_registry / sfr_log_model. Scratch: RUN_PARALLEL_DEMO <<- TRUE\n")
} else {
  reg <- sfr_model_registry(conn, database = ep_db, schema = ep_models)
  # mv <- sfr_log_model(reg, ...)
}


## Summary

- One YAML file for names + SPCS parameters.
- Driver blocks — monitor elsewhere.
